# Data Preprocessing

Encode categoricals, scale numerics, drop leakage/ID columns, and save the processed dataset.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib, yaml, os

with open('../configs/paths.yaml') as f:
    paths = yaml.safe_load(f)
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

os.makedirs(f'../{paths["data"]["processed"]}', exist_ok=True)
os.makedirs(f'../{paths["artifacts"]["models"]}', exist_ok=True)

df = pd.read_csv(f'../{paths["data"]["raw"]}')
print('Raw shape:', df.shape)
df.head()

Raw shape: (730, 16)


,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,01-01-2018,1,0,1,0,1,1,2,14.110847,18.18125,80.5833,10.749882,331,654,985
1,2,02-01-2018,1,0,1,0,2,1,2,14.902598,17.68695,69.6087,16.652113,131,670,801
2,3,03-01-2018,1,0,1,0,3,1,1,8.050924,9.47025,43.7273,16.636703,120,1229,1349
3,4,04-01-2018,1,0,1,0,4,1,1,8.200000,10.60610,59.0435,10.739832,108,1454,1562
4,5,05-01-2018,1,0,1,0,5,1,1,9.305237,11.46350,43.6957,12.522300,82,1518,1600


In [4]:
# ── 1. Parse date and derive basic date features
df['dteday'] = pd.to_datetime(df['dteday'], dayfirst=True)
df['day_of_year'] = df['dteday'].dt.dayofyear

In [5]:
# ── 2. Drop leakage and ID columns 
DROP_COLS = cfg['features']['drop_columns'] + ['dteday']
df_clean = df.drop(columns=DROP_COLS)
print('After drop:', df_clean.shape)
print('Columns:', df_clean.columns.tolist())

After drop: (730, 13)
Columns: ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt', 'day_of_year']


In [6]:
# ── 3. Verify no missing values 
print('Missing values:', df_clean.isnull().sum().sum())

Missing values: 0


In [7]:
# ── 4. Separate features and target 
TARGET = cfg['project']['target']
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]
print(f'X shape: {X.shape}  |  y shape: {y.shape}')

X shape: (730, 12)  |  y shape: (730,)


In [8]:
# ── 5. Scale numeric features 
NUM_COLS = cfg['features']['numeric']  # temp, atemp, hum, windspeed
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[NUM_COLS] = scaler.fit_transform(X[NUM_COLS])

print('Scaled columns:', NUM_COLS)
X_scaled[NUM_COLS].describe().round(3)

Scaled columns: ['temp', 'atemp', 'hum', 'windspeed']


,temp,atemp,hum,windspeed
count,730.000,730.000,730.000,730.000
mean,-0.000,0.000,-0.000,0.000
std,1.001,1.001,1.001,1.001
min,-2.385,-2.428,-4.411,-2.169
25%,-0.867,-0.839,-0.757,-0.717
50%,0.020,0.079,-0.010,-0.123
75%,0.875,0.825,0.719,0.551
max,2.001,2.249,2.424,4.090


In [11]:
# ── 6. Save scaler artifact 
scaler_path = f'../{paths["artifacts"]["models"]}scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved -> {scaler_path}')

Scaler saved -> ../models/scaler.pkl


In [12]:
# ── 7. Save processed dataset 
processed = X_scaled.copy()
processed[TARGET] = y.values
out_path = f'../{paths["data"]["processed"]}data_processed.csv'
processed.to_csv(out_path, index=False)
print(f'Processed data saved -> {out_path}')
print(processed.shape)
processed.head()

Processed data saved -> ../data/processed/data_processed.csv
(730, 13)


,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,day_of_year,cnt
0,1,0,1,0,1,1,2,-0.827613,-0.680818,1.252343,-0.387833,1,985
1,1,0,1,0,2,1,2,-0.722069,-0.741507,0.480996,0.748899,2,801
2,1,0,1,0,3,1,1,-1.635432,-1.750344,-1.338073,0.745931,3,1349
3,1,0,1,0,4,1,1,-1.615560,-1.610886,-0.261577,-0.389769,4,1562
4,1,0,1,0,5,1,1,-1.468226,-1.505615,-1.340294,-0.046477,5,1600


## Preprocessing Summary

| Step | Action |
|---|---|
| Date parse | `dteday` → datetime; derived `day_of_year` |
| Leakage drop | Removed `casual`, `registered`, `instant`, `dteday` |
| Missing values | None found |
| Scaling | StandardScaler on `temp`, `atemp`, `hum`, `windspeed` |
| Categoricals | Left as integer codes (tree models handle natively; OHE in NB 05) |

**Artifacts saved:** `models/scaler.pkl`, `data/processed/data_processed.csv`  
**Next:** `05_Feature_Engineering.ipynb`
